#### Imports

In [1]:
# Model Imports
from rfdetr import RFDETRMedium

# Additional Imports
import gc
import os
import sys
import torch
from zoneinfo import ZoneInfo
from datetime import datetime 

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Path Configurations

In [2]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Base Model and Project Path Configuration
base_model_name = "rfdetr_m"
resolution = 800
current_date = datetime.now(ZoneInfo("Europe/Berlin")).strftime("%d-%m-%Y_%H-%M")
project_name = f"{current_date}_{base_model_name.split('/')[-1]}"

# Path Configuration
data_directory = os.path.join(project_root , "data" , "detection")
dataset_name = "coco_football_players_detection_v11"
dataset_path = os.path.join(data_directory , dataset_name)

# Model Name and Weights
full_model_output_path = os.path.join(project_root , "models" , "detection", project_name)
os.makedirs(full_model_output_path, exist_ok=True)

#### Utils Imports

In [3]:
from utils.gpu import clear_gpu_memory

In [4]:
clear_gpu_memory()

GPU Memory Cleared
  Allocated : 0.0 MB → 0.0 MB
  Reserved  : 0.0 MB → 0.0 MB
  Peak      : 0.0 MB (reset)


{'before': {'allocated_mb': 0.0, 'reserved_mb': 0.0, 'peak_mb': 0.0},
 'after': {'allocated_mb': 0.0, 'reserved_mb': 0.0, 'peak_mb': 0.0}}

#### Model Setup

In [5]:
# Detection Model
detection_model = RFDETRMedium() # Can additionally load from a saved point

[2026-05-04 06:51:38] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-05-04 06:51:38] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-04 06:51:38] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-04 06:51:39] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


#### Data Augmentations

In [6]:
aug_config = {
    # Translate / slight rotate (ShiftScaleRotate equivalent)
    "Affine": {
        "translate_percent": (-0.1, 0.1),   # shift_limit=0.1
        "rotate": (-5.0, 5.0),              # rotate_limit=5
        "scale": (1.0, 1.0),               # scale_limit=0.0 (no scaling)
        "mode": 0,                          # border_mode=0 (constant)
        "p": 0.5,
    },
    # Flips
    "HorizontalFlip": {"p": 0.5},
    # HSV color augmentation
    "HueSaturationValue": {
        "hue_shift_limit": int(0.015 * 180),   # ~2
        "sat_shift_limit": int(0.7 * 255),     # ~178
        "val_shift_limit": int(0.4 * 255),     # ~102
        "p": 1.0,
    },
    # RandAugment equivalent: pick 2 of 7 transforms
    "SomeOf": {
        "transforms": [
            {"Equalize": {"p": 1.0}},
            {"Sharpen": {"p": 1.0}},
            {"RandomBrightnessContrast": {"p": 1.0}},
            {"Posterize": {"p": 1.0}},
            {"CLAHE": {"p": 1.0}},
            {"ColorJitter": {"p": 1.0}},
            {"Solarize": {"p": 1.0}},
        ],
        "n": 2,
        "p": 0.5,
    },
}

#### GPU Clearning

In [7]:
def on_epoch_end(data):
    gc.collect()
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved  = torch.cuda.memory_reserved()  / 1024**2
    print(f"Epoch end — Allocated: {allocated:.1f} MB | Reserved: {reserved:.1f} MB")

#### Model Training

In [8]:
detection_model.callbacks["on_fit_epoch_end"].append(on_epoch_end)

# Training Instructions
detection_model.train(
    dataset_dir=str(dataset_path),
    output_dir=str(full_model_output_path),
    epochs=150,
    batch=4,
    grad_accum_steps=4,      # keep effective batch size at 8
    lr=5e-5,
    fp16=True,
    early_stopping=True,
    early_stopping_patience=25,
    aug_config=aug_config,
    #resolution=resolution,           # 800/16=50 patches → meaningful memory drop vs 960
    num_workers=0,
    gradient_checkpointing=True,
)

[2026-05-04 06:51:40] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-04 06:51:40] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-04 06:51:40] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-05-04 06:51:41] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 4. The detection head will be re-initialized to 4 classes.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[2026-05-04 06:51:41] [INFO] rf-detr - Building Roboflow train dataset with square resize at resolution 576
[2026-05-04 06:51:41] [INFO] rf-detr - Using multi-scale training with square resize and scales: [736]
[2026-05-04 06:51:41] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-05-04 06:51:41] [INFO] rf-detr - Built 4 Albumentations transforms from config
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
[2026-05-04 06:51:41] [INFO] rf-detr - Building Roboflow val dataset with square resize at resolution 576
[2026-05-04 06:51:41] [INFO] rf-detr - Using multi-scale training with square resize and scales: [736]
[2026-05-04 06:51:41] [INFO] rf-detr - Built 1 Albumentations transforms from config


/opt/venv/lib/python3.12/site-packages/rfdetr/datasets/transforms.py:224: UserWarning: Argument(s) 'mode' are not valid for transform Affine
  return aug_cls(**_normalize_albu_params(name, params, aug_cls))
/opt/venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/tom/Desktop/Programming/Personal/live-footie-formations/models/detection/04-05-2026_08-51_rfdetr_m exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


...

                    Val — Per-class Metrics                     
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class      ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ball       │   0.2308 │ 0.2800 │ 0.7436 │    0.8788 │ 0.6444 │
│ goalkeeper │   0.6328 │ 0.6949 │ 0.9474 │    0.9730 │ 0.9231 │
│ player     │   0.7250 │ 0.7665 │ 0.9881 │    0.9927 │ 0.9836 │
│ referee    │   0.5926 │ 0.6453 │ 0.9177 │    0.9298 │ 0.9060 │
└────────────┴──────────┴────────┴────────┴───────────┴────────┘

Val — Overall Metrics                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.5481 │ 0.8763 │ 0.5858 │ 0.6025 │ 0.8961 │ 0.9372 │ 0.8643 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘

                    Val — Per-class Metrics                     
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class      ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ball       │   0.2316 │ 0.2867 │ 0.7436 │    0.8788 │ 0.6444 │
│ goalkeeper │   0.6341 │ 0.6949 │ 0.9351 │    0.9474 │ 0.9231 │
│ player     │   0.7236 │ 0.7671 │ 0.9881 │    0.9927 │ 0.9836 │
│ referee    │   0.6032 │ 0.6615 │ 0.9177 │    0.9298 │ 0.9060 │
└────────────┴──────────┴────────┴────────┴───────────┴────────┘

Monitored metric __rfdetr_effective_map__ did not improve in the last 25 records. Best score: 0.556. Signaling Trainer to stop.


[2026-05-04 08:15:36] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.5496, ema=0.5564)
